# Chapitre 5 - Cours 3 : réplication multi-leader et conflits

Avec un leader unique, toute écriture passe par un seul nœud : si on ne peut pas le
joindre, on ne peut pas écrire. Autoriser **plusieurs** nœuds à accepter des écritures
lève cette limite, mais fait apparaître le problème central de ce cours : des écritures
**concurrentes** sur des leaders différents, qu'il faut réconcilier. Il couvre la
section *Multi-Leader Replication* : *Use Cases for Multi-Leader Replication*, *Handling Write Conflicts* et *Multi-Leader Replication Topologies*.

**Plan.**

1. Plusieurs leaders : pourquoi seulement en asynchrone.
2. Le fonctionnement multi-centres de données : latence, pannes, cohérence.
3. Topologies de réplication et messages qui se doublent.
4. Les conflits d'écriture : d'où ils viennent, combien il y en a.
5. Éviter les conflits.
6. Le dernier écrivain gagne... et les autres disparaissent.
7. Garder les frères : résolution par l'application.
8. Converger vers un état cohérent.
9. Qu'est-ce qu'un conflit ?

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import replication as rp
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)

## 1. Plusieurs leaders (section *Multi-Leader Replication*)

Extension naturelle du leader unique : plusieurs nœuds acceptent les écritures, et chacun
transmet ses modifications à tous les autres ; chaque leader est en même temps suiveur des
autres. On parle aussi de réplication **active/active** ou **bidirectionnelle**.

Si la réplication entre leaders était **synchrone**, une écriture sur A attendrait B ; une
coupure entre A et B bloquerait les écritures sur A : on retrouverait exactement un leader
unique (B) auquel A ne ferait que relayer les requêtes. Tout l'intérêt est donc dans la
version **asynchrone**, où chaque leader traite ses écritures même coupé des autres.

Le multi-leader est souvent une fonction **rajoutée** à une base qui n'a pas été conçue
pour elle (MySQL, Oracle, SQL Server, extensions comme pglogical pour PostgreSQL) : les
clés auto-incrémentées, les déclencheurs et les contraintes d'intégrité y interagissent
mal. Beaucoup le considèrent comme un terrain dangereux, à éviter si possible.

> **À retenir.** Le multi-leader n'a de sens qu'en asynchrone : chaque leader écrit
> localement et propage plus tard. Le prix à payer : des écritures concurrentes à
> réconcilier.